# Inference for the baseline and four fine-tuned models

This cleaned release copy reads an existing authorized dataset's `test` split, selects its first 1,024 rows, and generates the comparison answers. It does not create or verify the upstream train/test partition.

Set `CAEE_DATASET_REPO` to an authorized dataset and `CAEE_FINETUNED_MODELS` to a JSON object mapping the four study suffixes to authorized model repository identifiers or local merged-model directories. Optional Hugging Face authentication uses `HF_TOKEN`. No private identifiers, credentials, model weights, or participant records are included.

Outputs are saved locally under `CAEE_INFERENCE_OUTPUT_DIR` (default: `outputs/inference`). They may contain participant information and must not be committed or published without a separate review. Dependency installation, authentication, and dataset/model loading may contact external services; this notebook does not send input records to a judge API or upload generated artifacts.

The original inference formatting, generation settings, answer parsing, ordering, and comparison-column mapping are retained. The notebook has been checked structurally and for Python syntax, not run on a GPU during repository preparation. Read `docs/REPRODUCIBILITY.md` before execution.


In [ ]:
!pip install -qqq -U pandas==2.2.2 transformers datasets accelerate trl bitsandbytes tqdm huggingface_hub pyarrow==16.1.0

In [ ]:
import gc
import os
import re
import torch
import transformers
import datasets
import shutil
import pandas as pd
from tqdm import tqdm
from datasets import Dataset, Value, Features, Sequence, load_dataset
from huggingface_hub import login, HfApi
from sklearn.model_selection import train_test_split
from trl import ORPOConfig, ORPOTrainer, setup_chat_format
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)


# Private locations and credentials are supplied by the execution environment.
import json
from pathlib import Path
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
inference_output_dir = Path(os.environ.get("CAEE_INFERENCE_OUTPUT_DIR", "outputs/inference"))
inference_output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Set torch dtype and attention implementation
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -qqq flash-attn
    torch_dtype = torch.bfloat16
    attn_implementation = "flash_attention_2"
else:
    torch_dtype = torch.float16
    attn_implementation = "eager"

# Load the test dataset from Hugging Face
dataset_name = os.environ.get("CAEE_DATASET_REPO")
if not dataset_name:
    raise ValueError("Set CAEE_DATASET_REPO to a dataset repository you are authorized to access.")
test_dataset = load_dataset(dataset_name, split="test")

# Set the number of rows selected for inference
num_rows_selected = 1024  # You can change this value dynamically (e.g., 512 or 1024)

# Select a subset of data for inference based on the num_rows_selected
test_prompts = test_dataset.select(range(num_rows_selected))

In [ ]:
# Function to format the messages into a single string
def format_messages(system_content, user_content):
    return f"system\n{system_content}\nuser\nPlease provide your response between 100 and 250 tokens and enclose the answer within double curly braces {{}}. {user_content}\nassistant\n"

# Function to extract text within curly braces, handling the first occurrence
def extract_text_within_curly_braces(text):
    start_idx = text.find('{')
    end_idx = text.find('}', start_idx)

    if start_idx != -1 and end_idx != -1:
        # Extract the text within the first curly braces
        return text[start_idx + 1:end_idx].strip()

    # If no curly braces found, return the entire text
    return text

# Function to perform inference for a given model and save the results
def perform_inference(model_name, num_rows_selected, suffix="base"):
    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=quant_config,
        offload_folder="offload"
    )

    # Set pad_token_id to eos_token_id for the tokenizer
    tokenizer.pad_token_id = tokenizer.eos_token_id

    # Initialize the pipeline with GPU support
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, model_kwargs={"torch_dtype": torch.bfloat16}, device_map="auto")

    # Prepare data for batch processing
    batch_size = 8
    inputs = [
        format_messages("You are a helpful AI assistant for user profiles on career trajectories analysis.", example["context"] + " " + example["question"])
        for example in test_prompts
    ]

    # Split inputs into batches
    input_batches = [inputs[i:i + batch_size] for i in range(0, len(inputs), batch_size)]

    # Perform inference with progress bar
    responses = []
    for batch in tqdm(input_batches, desc=f"Processing batches with {suffix} Model"):
        response = pipe(
            batch,
            max_new_tokens=256,
            batch_size=batch_size,
            pad_token_id=tokenizer.eos_token_id
        )
        responses.extend(response)

    # Process the responses
    results = []
    for i, response in enumerate(responses):
        formatted_messages = inputs[i]
        generated_text = response[0].get("generated_text", response[0].get("text", ""))
        model_answer = generated_text.replace(formatted_messages, "").strip()
        example = test_prompts[i]
        assistant_content = example["chosen"][-1]["content"] if example["chosen"][-1]["role"] == "assistant" else ""

        # Extract the text within double curly braces
        parsed_answer = extract_text_within_curly_braces(model_answer)

        results.append({
            "id": i,
            "prompt": example["prompt"],
            "question": example["question"],
            "context": example["context"],
            "system_content": "You are a helpful AI assistant for user profiles on career trajectories analysis.",
            "user_content": example["context"] + " " + example["question"],
            "expected_answer": assistant_content,
            f"{suffix}_model_answer": model_answer,
            f"parsed_answer_{suffix}": parsed_answer
        })

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    results_df.to_csv(inference_output_dir / f"{suffix}_model_inference_{num_rows_selected}.csv", index=False)
    print(f"{suffix.capitalize()} model inference results saved to {suffix}_model_inference_{num_rows_selected}.csv")

    # Flush memory before loading the next model
    del model, tokenizer, pipe
    gc.collect()
    torch.cuda.empty_cache()

    return results_df

suffixes = ["4096", "4096x1-25", "4096x1-50", "4096x1-75"]
model_locations_json = os.environ.get("CAEE_FINETUNED_MODELS")
if not model_locations_json:
    raise ValueError("Set CAEE_FINETUNED_MODELS to a JSON object mapping the four suffixes to authorized model locations.")
model_locations = json.loads(model_locations_json)
if not isinstance(model_locations, dict) or set(model_locations) != set(suffixes):
    raise ValueError("CAEE_FINETUNED_MODELS must contain exactly the four documented study suffixes.")
if not all(isinstance(model_locations[suffix], str) and model_locations[suffix].strip() for suffix in suffixes):
    raise ValueError("Each fine-tuned model location must be a nonempty string.")
finetuned_model_names = [model_locations[suffix] for suffix in suffixes]

# Perform inference for the base model
base_results_df = perform_inference("meta-llama/Meta-Llama-3-8B-Instruct", num_rows_selected, suffix="base")

# Perform inference for the fine-tuned models
finetuned_results_dfs = []
for model_name, suffix in zip(finetuned_model_names, suffixes):
    finetuned_results_df = perform_inference(model_name, num_rows_selected, suffix=suffix)
    finetuned_results_dfs.append(finetuned_results_df)

# Combine the results from the base model and fine-tuned models
comparison_df = base_results_df
for finetuned_df, suffix in zip(finetuned_results_dfs, suffixes):
    comparison_df = pd.merge(
        comparison_df,
        finetuned_df[["id", f"{suffix}_model_answer", f"parsed_answer_{suffix}"]],
        on="id"
    )

# Rename the columns for clarity
comparison_df.rename(columns={
    "model_answer": "model_answer_base",
    "parsed_answer": "parsed_answer_base"
}, inplace=True)

# Display the comparison
comparison_df = comparison_df[[
    "prompt",
    "question",
    "expected_answer",
    "parsed_answer_base",
    "parsed_answer_4096",
    "parsed_answer_4096x1-25",
    "parsed_answer_4096x1-50",
    "parsed_answer_4096x1-75"
]]
print({"comparison_rows": len(comparison_df), "comparison_columns": len(comparison_df.columns)})

# Save the combined comparison results to a CSV file
comparison_df.to_csv(inference_output_dir / f"inference_comparison_combined_{num_rows_selected}.csv", index=False)
print(f"Combined inference results saved locally for {num_rows_selected} rows")

# Flush memory
gc.collect()
torch.cuda.empty_cache()